# EVA — Yandex Cloud | 128-dim | 32 heads | 6 layers | 1.2M params

**Перед запуском:**
1. `git clone https://github.com/BlackCatSpb/FCF.git && cd FCF`
2. `pip install -r requirements.txt`
3. Загрузить `connected_ru.npy` в `real_data/` (вручную через интерфейс VM)
4. Загрузить `evolved_affinity.pt` в `checkpoints/symbolic/`
5. Запустить ячейки по порядку 0 → 1 → 2 → 3 → ...

In [ ]:
# 0. Клонировать репозиторий, установить пакеты, найти данные
import os, sys, subprocess

# Clone repo if not already here
if not os.path.exists('train_yandex.py'):
    print('Cloning FCF...')
    subprocess.run(['git', 'clone', 'https://github.com/BlackCatSpb/FCF.git'])
    os.chdir('FCF')

print(f'CWD: {os.getcwd()}')
os.makedirs('real_data', exist_ok=True)
os.makedirs('checkpoints/symbolic', exist_ok=True)

# Install dependencies
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'torch', 'numpy', 'scikit-learn', 'loguru', 'psutil'])

print('Setup done.')

In [ ]:
# 1. Проверка GPU и данных
import torch, numpy as np, os

print(f'PyTorch {torch.__version__}')
if torch.cuda.is_available():
    g = torch.cuda.get_device_properties(0)
    print(f'GPU: {g.name} ({g.total_memory/1e9:.1f} GB VRAM)')
else:
    print('WARNING: CUDA not available!')

npy = None
for p in ['real_data/connected_ru.npy', 'connected_ru.npy', 'real_data/full_corpus_ids.npy']:
    if os.path.exists(p):
        d = np.load(p, mmap_mode='r').astype(np.int32)
        npy = p
        print(f'Corpus: {p} ({len(d)/1e6:.1f}M tokens)')
        break
if not npy:
    print('FATAL: connected_ru.npy not found!')

ckpt = os.listdir('checkpoints/symbolic')
print(f'Checkpoints: {len(ckpt)} files')
need = 'evolved_affinity.pt'
if need not in ckpt:
    print(f'WARNING: {need} missing — train_word_pipeline.py needed first')
else:
    print(f'  {need}: OK')

In [ ]:
# 2. Обучение (200K шагов, ~6-8 часов)
import subprocess, sys
subprocess.run([sys.executable, 'train_yandex.py'])

In [ ]:
# 3. Статус (запускать пока идёт обучение)
import os
log = 'yandex_train_log.txt'
if os.path.exists(log):
    lines = open(log).readlines()
    print(f'Lines: {len(lines)}')
    for l in lines[-15:]:
        print(l.rstrip())
else:
    print('Ждите 2-3 минуты — лог появится после первого сохранения.')

c = 'checkpoints/symbolic'
ckpts = sorted([f for f in os.listdir(c) if f.startswith('yandex_')])
if ckpts:
    print(f'Сохранено чекпоинтов: {len(ckpts)}')

In [ ]:
# 4. Тест генерации
import torch, torch.nn.functional as F, os, sys
sys.path.insert(0, os.getcwd())
from eva.symbolic.char_vocab import CharacterVocab
from eva.symbolic.unified_transformer import UnifiedMultidimensionalTransformer

cv = CharacterVocab(); DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
ut = UnifiedMultidimensionalTransformer(vocab_size=157, coord_dim=128,
    num_levels=8, scales_per_level=4, num_layers=6, d_ff=512).to(DEVICE)

ckpt_path = 'checkpoints/symbolic/yandex_latest.pt'
if not os.path.exists(ckpt_path):
    print(f'{ckpt_path} not found — training still running?')
else:
    ckpt = torch.load(ckpt_path, map_location='cpu')
    ut.load_state_dict(ckpt['ut'], strict=False)
    ut.eval()

    ev = torch.load('checkpoints/symbolic/evolved_affinity.pt', map_location='cpu')
    c = ev['coords'].to(DEVICE); c128 = torch.zeros(157, 128, device=DEVICE)
    c128[:, :24] = c[:, :24]
    g = torch.Generator(device=DEVICE).manual_seed(42)
    c128[:, 24:] = torch.randn(157, 104, generator=g, device=DEVICE) * 0.02
    c128 = c128 / c128.norm(dim=-1, keepdim=True).clamp(1e-8)
    ut.set_symbol_coordinates(c128)

    def gen(ids, n=30, T=0.8):
        ids = list(ids)
        with torch.no_grad():
            for _ in range(n):
                _, sc = ut(torch.tensor([ids], dtype=torch.long, device=DEVICE), return_scores=True)
                logits = sc[0, -1] / T
                sl, si = logits.sort(descending=True)
                cp = F.softmax(sl, dim=-1).cumsum(dim=-1)
                cut = (cp > 0.95).nonzero(as_tuple=True)[0]
                k = cut[0].item() + 1 if len(cut) > 0 else 30
                v, idx = logits.topk(min(k, 50)); p = F.softmax(v, dim=-1)
                for t in set(ids[-5:]):
                    m = (idx == t).nonzero(as_tuple=True)[0]
                    if len(m) > 0: p[m] *= 0.2
                p /= p.sum(); nt = idx[torch.multinomial(p, 1)].item()
                if nt <= 0 or nt >= 157: nt = idx[0].item()
                ids.append(nt)
        return ids

    for w in ['привет','человек идет','солнце светит','сегодня хорошая','я люблю','метаданные хранят']:
        ids = cv.encode(w)[1:-1]
        if len(ids) >= 2:
            r = gen(ids, 30, 0.8)
            print(f'{w} -> {cv.decode(r)}')

In [ ]:
# 5. Сохранить результаты
import os, shutil, json
os.makedirs('export', exist_ok=True)

for f in os.listdir('checkpoints/symbolic'):
    if f.startswith('yandex_'):
        shutil.copy2(f'checkpoints/symbolic/{f}', f'export/{f}')
        print(f'  {f}')

log = 'yandex_train_log.txt'
if os.path.exists(log):
    shutil.copy2(log, f'export/{log}')

summary = {'files': os.listdir('export'), 'params': '128-dim, 32 heads, 6 layers'}
with open('export/summary.json', 'w') as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

import subprocess, sys
subprocess.run([sys.executable, '-c', '''
import shutil
shutil.make_archive("eva_trained", "zip", "export")
'''])

print('Done. Download eva_trained.zip')
for f in os.listdir('.'):
    if f.endswith('.zip'):
        print(f'  {f}: {os.path.getsize(f)/1e6:.1f} MB')